In [21]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [27]:
df = pd.read_csv("/Users/bharathkumar/Documents/Stock Price Prediction/ARIMA/us_stock_data_2years_with_cleaned.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

series = df['Close'].astype(float)
series = series.dropna()

In [ ]:
exog_features = ['SMA_20','EMA_20','RSI_14','BB_Middle','BB_Upper','BB_Lower','Volume']
X = df[exog_features]

X = df[exog_features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(method='ffill').fillna(method='bfill')

y = df['Close']

In [29]:
train_size = int(len(series) * 0.8)
y_train, y_test = y[:train_size], y[train_size:]
X_train, X_test = X[:train_size], X[train_size:]

In [30]:
model = SARIMAX(y_train, exog=X_train, order=(1,1,1), enforce_stationarity=False, enforce_invertibility=False)
model_fit = model.fit(disp=False)

In [33]:
train_pred = model_fit.predict(start=y_train.index[0], end=y_train.index[-1], exog=X_train)

test_pred = model_fit.predict(start=y_test.index[0], end=y_test.index[-1], exog=X_test)

def clean_pair(a,b):
    df_clean = pd.DataFrame({"true": a, "pred": b})
    df_clean = df_clean.replace([np.inf,-np.inf], np.nan).dropna()
    return df_clean["true"], df_clean["pred"]

y_train_clean, train_pred = clean_pair(y_train, train_pred)
y_test_clean, test_pred = clean_pair(y_test, test_pred)

def directional_accuracy(true, pred):
    true_dir = np.sign(true.diff().dropna())
    pred_dir = np.sign(pred.diff().dropna())
    return (true_dir == pred_dir).mean()

train_r2 = r2_score(y_train_clean, train_pred)
train_mae = mean_absolute_error(y_train_clean, train_pred)
train_da = directional_accuracy(y_train_clean, train_pred)


print(f"R²: {train_r2:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"Directional Accuracy: {train_da*100:.2f}%")


R²: 0.9990
MAE: 5.6224
Directional Accuracy: 64.16%
